# 線上零售 RFM 分析 II

Kaggle 資料集：
https://www.kaggle.com/code/ekrembayar/rfm-analysis-online-retail-ii/notebook

本 Notebook 主要步驟：

- 資料清理
- RFM 計算（Recency、Frequency、Monetary）
- 顧客分群
- 可視化分析
- 列印顧客 FPD 資訊（頻率、金額、最近消費日）

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 載入資料集

In [ ]:
file_path = 'online_retail_II.xlsx'

xls = pd.ExcelFile(file_path)
print(xls.sheet_names)  # 顯示所有工作表名稱

df_list = []

for sheet in xls.sheet_names:
    temp = pd.read_excel(file_path, sheet_name=sheet)
    temp['SheetName'] = sheet
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)
df.head()

## 資料清理

In [ ]:
# 將欄位名稱轉為無空格並以底線連接
df.columns = [c.strip().replace(' ','_') for c in df.columns]

# 將日期欄位轉為 datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 移除缺失 Customer_ID 的資料
df = df.dropna(subset=['Customer_ID'])

# 移除退貨單據（Invoice 以 C 開頭）
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# 只保留數量與價格皆大於 0 的資料
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# 計算總金額
df['TotalPrice'] = df['Quantity'] * df['Price']

## 建立 RFM 表格

In [ ]:
# 分析基準日期為資料最後一日 + 2 天
analysis_date = df['InvoiceDate'].max() + timedelta(days=2)

# 計算 RFM
rfm = df.groupby('Customer_ID').agg({
    'InvoiceDate': lambda x: (analysis_date - x.max()).days,
    'Invoice':'nunique',
    'TotalPrice':'sum'
})

rfm.columns = ['recency','frequency','monetary']
rfm.head()

## 計算 RFM 分數

In [ ]:
rfm['recency_score'] = pd.qcut(rfm['recency'],5,labels=[5,4,3,2,1]).astype(int)

rfm['frequency_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    5,
    labels=[1,2,3,4,5]
).astype(int)

rfm['monetary_score'] = pd.qcut(rfm['monetary'],5,labels=[1,2,3,4,5]).astype(int)

rfm.head()

## 顧客分群

In [ ]:
rfm['rfm_segment'] = rfm['recency_score'].astype(str) + rfm['frequency_score'].astype(str)

seg_map = {
 r"[1-2][1-2]":"沉睡客戶",
 r"[1-2][3-4]":"風險客戶",
 r"[1-2]5":"不能失去客戶",
 r"3[1-2]":"即將流失",
 r"33":"需要關注",
 r"[3-4][4-5]":"忠誠客戶",
 r"41":"潛力客戶",
 r"51":"新客戶",
 r"[4-5][2-3]":"潛在忠誠",
 r"5[4-5]":"明星客戶"
}

rfm['segment'] = rfm['rfm_segment'].replace(seg_map, regex=True)
rfm.head()

## 可視化分析

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(data=rfm.reset_index(),x='segment',order=rfm['segment'].value_counts().index)
plt.xticks(rotation=45)
plt.title('各顧客分群數量')
plt.show()

## 列印 FPD（頻率 Frequency、金額 Monetary、最近消費日 Recency）

In [ ]:
rfp = rfm.copy()
rfp['last_purchase'] = analysis_date - pd.to_timedelta(rfp['recency'], unit='d')
rfp_display = rfp[['frequency','monetary','last_purchase','segment']].reset_index()

print('前 10 筆顧客 FPD 資訊：')
print(rfp_display.head(10))